In [13]:

# Notebook-ready script
# ---------------------
# Reads a CSV with model-selection results and generates two LaTeX booktabs tables:
#   1) LLM mapper
#   2) MultiStep mapper
#
# Layout: datasets as row groups, metrics as columns.
# Rows = models, Groups = datasets, Columns = 5 metrics.

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# ============================================================
# 1) CONFIGURATION YOU MAY WANT TO EDIT
# ============================================================

CSV_PATH = "results_summary_all.csv"   # change if needed
ONLY_SUCCESS = True
DECIMALS = 2

# Which metrics to show, in which order
# Two-layer framing: syntax (exact match + predicate overlap) then semantics (violations)
# Use "\n" inside the display name to split column headers across two lines (\shortstack).
METRICS = [
    ("constraint_exact_f1",       "Exact Predicate\nMatch-F1"),
    ("pairwise_syn_jaccard_mean",  "Predicate\nJaccard"),
    ("agg_viol_f1",               "Viol.-\nF1"),
    ("agg_viol_recall",           "Viol.-\nRecall"),
    ("pairwise_viol_f1_mean",     "Pairwise.\nViol.-F1"),
]

# Order of implementations in output
IMPLEMENTATION_ORDER = ["llm", "multi_step_refinement"]

# Order of datasets in output; any dataset not listed here will be appended alphabetically
DATASET_ORDER = [
   
    "hospital",
    "restaurant",
    "sisu",
     "tax"
]

# Order of models in output; any model not listed here will be appended alphabetically
MODEL_ORDER = [
    "mistral-small-2506",
    "mistral-large-2512",
    "gpt-5-mini",
    "gpt-5",
    "llama-3.3-70b-versatile",
]

# Pretty names used in the table rows
MODEL_DISPLAY = {
    "mistral-small-2506":      "Mistral-Small",
    "mistral-large-2512":      "Mistral-Large",
    "llama-3.3-70b-versatile": "Llama 3.3",
    "gpt-5-mini":              "GPT-5 mini",
    "gpt-5":                   "GPT-5",
}

# Pretty dataset names used in row-group headers
DATASET_DISPLAY = {
    "tax":        "Tax",
    "hospital":   "Hospital",
    "restaurant": "Restaurant",
    "sisu":       "SISU",
}

# Pretty implementation names used in captions / labels / headings
IMPLEMENTATION_DISPLAY = {
    "llm": "LLM",
    "multi_step_refinement": "MultiStep",
}

# Optional: if you want to override row grouping by implementation based on variant_name
IMPLEMENTATION_VARIANT_HINTS = {
    "LLM": "llm",
    "MultiStep": "multi_step_refinement",
}

# LaTeX labels
LABELS = {
    "llm": "tab:model-selection-llm",
    "multi_step_refinement": "tab:model-selection-multistep",
}

# Captions
CAPTIONS = {
    "llm": (
        "Model-selection results for the \\textsc{LLM} mapper. "
        "Rows are grouped by dataset. "
        "Syntax layer: DC Exact-Match F1 and DC Predicate Jaccard. "
        "Semantics layer: Violation F1 (primary), Violation Recall, and Sent.\\ Violation F1. "
        "Bold = best per group; dash = no valid run."
    ),
    "multi_step_refinement": (
        "Model-selection results for the \\textsc{MultiStep} mapper. "
        "Rows are grouped by dataset. "
        "Syntax layer: DC Exact-Match F1 and DC Predicate Jaccard. "
        "Semantics layer: Violation F1 (primary), Violation Recall, and Sent.\\ Violation F1. "
        "Bold = best per group; dash = no valid run."
    ),
}

# If True, bold the best value within each (dataset, metric) column
BOLD_BEST = True

# If True, underline the second-best value within each (dataset, metric) column
UNDERLINE_SECOND = False

# If True, escape underscores etc. in model/dataset names
ESCAPE_LATEX = True


# ============================================================
# 2) SMALL UTILITIES
# ============================================================

def escape_latex(s: str) -> str:
    if not ESCAPE_LATEX:
        return str(s)
    s = str(s)
    repl = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for k, v in repl.items():
        s = s.replace(k, v)
    return s

def canonicalize_implementation(row):
    impl = str(row.get("implementation", "")).strip()
    if impl in IMPLEMENTATION_DISPLAY:
        return impl
    variant_name = str(row.get("variant_name", "")).strip()
    for hint, canonical in IMPLEMENTATION_VARIANT_HINTS.items():
        if hint.lower() in variant_name.lower():
            return canonical
    return impl

def ordered_unique(values, preferred_order):
    values = list(dict.fromkeys(values))
    preferred = [x for x in preferred_order if x in values]
    rest = sorted([x for x in values if x not in preferred])
    return preferred + rest

def metric_header(name: str) -> str:
    """Wrap multi-line metric names in \\shortstack[r]{} for compact column headers."""
    if '\n' in name:
        parts = name.split('\n')
        inner = r" \\ ".join(parts)
        return rf"\shortstack[r]{{{inner}}}"
    return name

def format_num(x, decimals=3):
    if pd.isna(x):
        return "-"
    return f"{float(x):.{decimals}f}"

def compute_best_and_second(df_impl, datasets, metrics):
    best_map = {}
    second_map = {}
    for d in datasets:
        for metric, _ in metrics:
            sub = df_impl[df_impl["dataset"] == d][["model", metric]].dropna()
            if sub.empty:
                best_map[(d, metric)] = set()
                second_map[(d, metric)] = set()
                continue
            vals = sorted(sub[metric].astype(float).unique(), reverse=True)
            best_val = vals[0]
            second_val = vals[1] if len(vals) > 1 else None
            best_map[(d, metric)] = set(sub.loc[sub[metric].astype(float) == best_val, "model"])
            second_models = set()
            if second_val is not None:
                second_models = set(sub.loc[sub[metric].astype(float) == second_val, "model"])
            second_map[(d, metric)] = second_models
    return best_map, second_map

def stylize_value(model, dataset, metric, value, best_map, second_map, decimals=3):
    text = format_num(value, decimals)
    if text == "-":
        return text
    if BOLD_BEST and model in best_map.get((dataset, metric), set()):
        text = rf"\textbf{{{text}}}"
    elif UNDERLINE_SECOND and model in second_map.get((dataset, metric), set()):
        text = rf"\underline{{{text}}}"
    return text


# ============================================================
# 3) LOAD DATA
# ============================================================

csv_path = Path(CSV_PATH)
if not csv_path.exists():
    raise FileNotFoundError(
        f"CSV not found at: {csv_path}\n"
        f"Update CSV_PATH at the top of the notebook."
    )

df = pd.read_csv(csv_path)

df["implementation_canonical"] = df.apply(canonicalize_implementation, axis=1)

if ONLY_SUCCESS and "status" in df.columns:
    df = df[df["status"].astype(str).str.lower() == "success"].copy()

df = df[df["implementation_canonical"].isin(IMPLEMENTATION_ORDER)].copy()

dups = df.groupby(["dataset", "implementation_canonical", "model"]).size().reset_index(name="n")
bad = dups[dups["n"] > 1]
if not bad.empty:
    display(Markdown("**Warning:** duplicate rows found. Keeping first occurrence."))
    if "runtime_s" in df.columns:
        df = df.sort_values(["dataset", "implementation_canonical", "model", "runtime_s"])
    df = df.drop_duplicates(subset=["dataset", "implementation_canonical", "model"], keep="first").copy()


# ============================================================
# 4) ORDER DATASETS AND MODELS
# ============================================================

datasets = ordered_unique(df["dataset"].tolist(), DATASET_ORDER)
models   = ordered_unique(df["model"].tolist(),   MODEL_ORDER)

dataset_display = {d: DATASET_DISPLAY.get(d, d.replace("_", " ").title()) for d in datasets}
model_display   = {m: MODEL_DISPLAY.get(m, m) for m in models}


# ============================================================
# 5) LATEX TABLE GENERATOR  (datasets as row groups)
# ============================================================

def generate_booktabs_table(df_impl, impl_key, datasets, models, metrics):
    """
    Build a LaTeX booktabs table where datasets become row groups and
    metrics are the columns. Missing (model, dataset) cells render as "-".
    """
    impl_name = IMPLEMENTATION_DISPLAY.get(impl_key, impl_key)
    n_cols = 2 + len(metrics)   # Dataset column + Model column + one per metric

    best_map, second_map = compute_best_and_second(df_impl, datasets, metrics)

    col_spec = "ll" + "r" * len(metrics)

    lines = []
    lines.append(r"\begin{table}[ht!]")
    lines.append(r"\centering")
    lines.append(r"\setlength{\tabcolsep}{4pt}")
    lines.append(rf"\caption{{{CAPTIONS.get(impl_key, impl_name)}}}")
    lines.append(rf"\label{{{LABELS.get(impl_key, 'tab:TODO')}}}")
    lines.append(r"\resizebox{\columnwidth}{!}{%")
    lines.append(rf"\begin{{tabular}}{{{col_spec}}}")
    lines.append(r"\toprule")

    # Header: Dataset | Model | metric columns
    header = [r"\textbf{Dataset}", r"\textbf{Model}"]
    for _, short_name in metrics:
        header.append(rf"\textbf{{{metric_header(short_name)}}}")
    lines.append(" & ".join(header) + r" \\")
    lines.append(r"\midrule")

    df_impl_idx = df_impl.set_index(["model", "dataset"])

    for i, d in enumerate(datasets):
        dset_name = escape_latex(dataset_display[d])
        n_models = len(models)

        for j, model in enumerate(models):
            model_name = escape_latex(model_display[model])
            # First row of group: emit \multirow; subsequent rows: empty cell
            if j == 0:
                dataset_col = rf"\multirow{{{n_models}}}{{*}}{{\textit{{{dset_name}}}}}"
            else:
                dataset_col = ""
            row = [dataset_col, rf"\textsc{{{model_name}}}"]
            for metric, _ in metrics:
                if (model, d) in df_impl_idx.index:
                    value = df_impl_idx.loc[(model, d), metric]
                    if isinstance(value, pd.Series):
                        value = value.iloc[0]
                else:
                    value = np.nan
                cell = stylize_value(
                    model=model,
                    dataset=d,
                    metric=metric,
                    value=value,
                    best_map=best_map,
                    second_map=second_map,
                    decimals=DECIMALS,
                )
                row.append(cell)
            lines.append(" & ".join(row) + r" \\")

        if i < len(datasets) - 1:
            lines.append(r"\midrule")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}}")
    lines.append(r"\end{table}")

    return "\n".join(lines)


# ============================================================
# 6) GENERATE THE TWO TABLES
# ============================================================

latex_tables = {}

for impl in IMPLEMENTATION_ORDER:
    df_impl = df[df["implementation_canonical"] == impl].copy()
    if df_impl.empty:
        continue
    latex_tables[impl] = generate_booktabs_table(
        df_impl=df_impl,
        impl_key=impl,
        datasets=datasets,
        models=models,
        metrics=METRICS,
    )

for impl in IMPLEMENTATION_ORDER:
    if impl in latex_tables:
        display(Markdown(f"## {IMPLEMENTATION_DISPLAY.get(impl, impl)} table"))
        print(latex_tables[impl])
        print()


# ============================================================
# 7) SAVE TO .TEX FILES
# ============================================================

OUTPUT_DIR = Path("latex_tables")
OUTPUT_DIR.mkdir(exist_ok=True)

for impl, latex in latex_tables.items():
    out_path = OUTPUT_DIR / f"{impl}_model_selection_table.tex"
    out_path.write_text(latex, encoding="utf-8")

display(Markdown(f"Saved `.tex` files to `{OUTPUT_DIR}`"))


# ============================================================
# 8) ALL TABLES TOGETHER FOR COPY/PASTE
# ============================================================

all_tables = "\n\n".join(latex_tables[impl] for impl in IMPLEMENTATION_ORDER if impl in latex_tables)

display(Markdown("## All tables together"))
print(all_tables)


## LLM table

\begin{table}[ht!]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Model-selection results for the \textsc{LLM} mapper. Rows are grouped by dataset. Syntax layer: DC Exact-Match F1 and DC Predicate Jaccard. Semantics layer: Violation F1 (primary), Violation Recall, and Sent.\ Violation F1. Bold = best per group; dash = no valid run.}
\label{tab:model-selection-llm}
\resizebox{\columnwidth}{!}{%
\begin{tabular}{llrrrrr}
\toprule
\textbf{Dataset} & \textbf{Model} & \textbf{\shortstack[r]{Exact Predicate \\ Match-F1}} & \textbf{\shortstack[r]{Predicate \\ Jaccard}} & \textbf{\shortstack[r]{Viol.- \\ F1}} & \textbf{\shortstack[r]{Viol.- \\ Recall}} & \textbf{\shortstack[r]{Pairwise. \\ Viol.-F1}} \\
\midrule
\multirow{5}{*}{\textit{Hospital}} & \textsc{Mistral-Small} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00} \\
 & \textsc{Mistral-Large} & 0.80 & 0.87 & 0.91 & \textbf{1.00} & 0.87 \\
 & \textsc{GPT-5 mini} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00}

## MultiStep table

\begin{table}[ht!]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Model-selection results for the \textsc{MultiStep} mapper. Rows are grouped by dataset. Syntax layer: DC Exact-Match F1 and DC Predicate Jaccard. Semantics layer: Violation F1 (primary), Violation Recall, and Sent.\ Violation F1. Bold = best per group; dash = no valid run.}
\label{tab:model-selection-multistep}
\resizebox{\columnwidth}{!}{%
\begin{tabular}{llrrrrr}
\toprule
\textbf{Dataset} & \textbf{Model} & \textbf{\shortstack[r]{Exact Predicate \\ Match-F1}} & \textbf{\shortstack[r]{Predicate \\ Jaccard}} & \textbf{\shortstack[r]{Viol.- \\ F1}} & \textbf{\shortstack[r]{Viol.- \\ Recall}} & \textbf{\shortstack[r]{Pairwise. \\ Viol.-F1}} \\
\midrule
\multirow{5}{*}{\textit{Hospital}} & \textsc{Mistral-Small} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00} \\
 & \textsc{Mistral-Large} & 0.93 & 0.98 & 0.56 & 0.39 & 0.94 \\
 & \textsc{GPT-5 mini} & \textbf{1.00} & \textbf{1.00} & \textbf{1.

Saved `.tex` files to `latex_tables`

## All tables together

\begin{table}[ht!]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Model-selection results for the \textsc{LLM} mapper. Rows are grouped by dataset. Syntax layer: DC Exact-Match F1 and DC Predicate Jaccard. Semantics layer: Violation F1 (primary), Violation Recall, and Sent.\ Violation F1. Bold = best per group; dash = no valid run.}
\label{tab:model-selection-llm}
\resizebox{\columnwidth}{!}{%
\begin{tabular}{llrrrrr}
\toprule
\textbf{Dataset} & \textbf{Model} & \textbf{\shortstack[r]{Exact Predicate \\ Match-F1}} & \textbf{\shortstack[r]{Predicate \\ Jaccard}} & \textbf{\shortstack[r]{Viol.- \\ F1}} & \textbf{\shortstack[r]{Viol.- \\ Recall}} & \textbf{\shortstack[r]{Pairwise. \\ Viol.-F1}} \\
\midrule
\multirow{5}{*}{\textit{Hospital}} & \textsc{Mistral-Small} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00} \\
 & \textsc{Mistral-Large} & 0.80 & 0.87 & 0.91 & \textbf{1.00} & 0.87 \\
 & \textsc{GPT-5 mini} & \textbf{1.00} & \textbf{1.00} & \textbf{1.00}